# Step 9 · Collect human preferences

Two answers to the same question → a person picks the better one → (question, chosen, rejected).

*Same as the dashboard page, done in code.*

In [ ]:
# ── Setup (run first) ────────────────────────────────────────────────────────────
# Uses the same lab registry as the dashboard. To experiment in a sandbox instead:
#   os.environ["LAB_HOME"] = "/tmp/my_lab"   (before importing lab)
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT); os.chdir(ROOT)
os.environ.setdefault("HF_HUB_OFFLINE", "1")          # models are already cached locally
import pandas as pd
from lab import data, registry, paths, rewards
data.bootstrap_samples()
BASE = registry.ensure_base(paths.DEFAULT_BASE_MODEL)
if not registry.rewards():
    registry.add_reward(rewards.DEFAULT_RULES, note="Default draft rules")

def latest(kind):
    """Newest model id of a kind: base | sft | rl | rlhf | rm (or None)."""
    ms = [m for m in registry.models() if m["kind"] == kind]
    return ms[-1]["id"] if ms else None

def latest_ds(role):
    d = registry.latest_dataset(role)
    return d["id"] if d else None

print("Lab home:", paths.home(), "| base model:", BASE)

In [ ]:
SAVE = False   # set True to write new versions into your lab (they appear in the dashboard)

In [ ]:
from lab import engines
MODEL = latest("sft") or BASE
eng = engines.get(registry.model(MODEL)["base_model"])
questions = [r["prompt"] for r in data.load(latest_ds("rl_train"))][:3]
pairs = []
for i, q in enumerate(questions):
    a = eng.generate(MODEL, [q], temperature=0.9, seed=11 + i)[0]
    b = eng.generate(MODEL, [q], temperature=0.9, seed=17 + i)[0]
    pairs.append({"prompt": q, "A": a, "B": b})
pd.DataFrame(pairs)

In [ ]:
# YOUR choice goes here. (This demo picks the answer that mentions checking — replace with your judgement.)
choices = ["A" if "check" in p["A"].lower() else "B" for p in pairs]
prefs = [{"prompt": p["prompt"], "chosen": p[c], "rejected": p["B" if c == "A" else "A"]} for p, c in zip(pairs, choices)]
pd.DataFrame(prefs)

In [ ]:
if SAVE:
    base = latest_ds("pref"); old = data.load(base)
    rows = old + [dict(p, id=f"pref-nb{len(old) + i + 1:03d}") for i, p in enumerate(prefs)]
    print(data.save_version("pref", rows, parent_id=base, source="human feedback", note="from notebook"))
print("PREF dataset:", latest_ds("pref"), registry.latest_dataset("pref")["n_rows"], "pairs")